## Example of the `aitlas` toolbox in the context of image segmentation
---
```
Author: Nejc Čož
Organisation: ZRC SAZU
Ljubljana, 2025
```
---

### Importing required packages

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
#from aitlas.datasets import TiiLIDARDatasetSegmentation
from aitlas.models import HRNet
#model_config = TiiLIDARDatasetSegmentation.get_fixed_model_config()

In [3]:
# --- imports you need at the top of the file ---
import os
import glob
from collections import defaultdict
from typing import Optional, Dict

import numpy as np
import rasterio
import torch

from aitlas.utils import image_loader
from aitlas.datasets.semantic_segmentation import SemanticSegmentationDataset
from aitlas.datasets.schemas import TiiLIDARDatasetBinaryWithPreprocessingSchema

# ----------------------------------------------------------------------
# Your dataset
# ----------------------------------------------------------------------
class TiiLIDARDatasetSegmentation(SemanticSegmentationDataset):
    schema = TiiLIDARDatasetBinaryWithPreprocessingSchema
    url = ""

    labels = ["Background", "barrow"]
    color_mapping = [[0, 0, 0], [255, 255, 255]]
    name = "TII LIDAR Binary"

    @staticmethod
    def get_fixed_model_config():
        return {
            "num_classes": 2,
            "learning_rate": 1e-4,
            "pretrained": True,
            "use_cuda": torch.cuda.is_available(),
            "threshold": 0.5,
            "metrics": ["iou"],
        }

    def __init__(self, config):
        super().__init__(config)  # validate schema/splits upstream
        self.images: list[str] = []
        self.masks: list[str] = []
        self.load_dataset(self.config.data_dir)

    # ---------- helpers: matching by two-part prefix ----------

    @staticmethod
    def _two_part_prefix(filename: str) -> str:
        """
        Extract the match key as the first two '__'-separated parts.
        Example: 'A__B__anything_else.tif' -> 'A__B'
        """
        parts = filename.split("__")
        return f"{parts[0]}__{parts[1]}" if len(parts) >= 2 else ""

    @staticmethod
    def _build_image_index(images_dir: str, prefer_visualisation: Optional[str]) -> Dict[str, str]:
        """
        Scan the images directory once and build an index:
            key = first two '__' parts (e.g., 'A__B')
            value = best-matching image path for that key
        If multiple images share a key, prefer those whose filename contains
        the requested visualisation token (e.g., 'slrm'); else pick a stable default.
        """
        idx = defaultdict(list)

        # Collect candidate image files (.tif/.tiff, any case)
        for ext in ("*.tif", "*.tiff", "*.TIF", "*.TIFF"):
            for p in glob.glob(os.path.join(images_dir, ext)):
                key = TiiLIDARDatasetSegmentation._two_part_prefix(os.path.basename(p))
                if key:
                    idx[key].append(p)

        # Choose the "best" path per key
        best = {}
        pref = (prefer_visualisation or "").lower()
        for key, paths in idx.items():
            if pref:
                cand = [p for p in paths if pref in os.path.basename(p).lower()]
                if cand:
                    best[key] = sorted(cand)[0]
                    continue
            # Fallback: deterministic choice: shortest filename, then lexicographic
            best[key] = sorted(
                paths,
                key=lambda p: (len(os.path.basename(p)), os.path.basename(p).lower())
            )[0]
        return best

    # ---------- mask reading (AO-aware), simplified signature ----------

    def process_single_mask(self, mask_path: str) -> np.ndarray:
        """
        Convert a multi-band TIFF mask into a binary 2D array (H, W) with values {0,1}.
        - If self.object_class == 'AO': union (logical OR) of up to first 3 class bands.
        - Else: read only band (self.object_class_band_id + 1).
        - Keep only pixels whose DFM value is in self.DFM_quality (e.g., [1]).
        """
        with rasterio.open(mask_path) as src:
            if getattr(self, "object_class", None) == "AO":
                nb = min(3, src.count)
                arr = src.read(indexes=list(range(1, nb + 1)))         # (B, H, W)
                mask_bool = np.isin(arr, self.DFM_quality).any(axis=0) # (H, W) bool
            else:
                band = src.read(self.object_class_band_id + 1)         # (H, W)
                mask_bool = np.isin(band, self.DFM_quality)            # (H, W) bool

        return mask_bool.astype(np.uint8)  # {0,1}

    # ---------- the pair building (index-based) ----------

    def load_dataset(self, data_dir):
        """
        Build the dataset by pairing masks with images using the first two '__' parts.
        Assumes every mask is valid (contains at least one positive), so we SKIP
        per-mask filtering for speed/simplicity.
        """
        # Cache config
        self.object_class = self.config.object_class                   # e.g., 'barrow' or 'AO'
        self.object_class_band_id = self.config.object_class_band_id   # 0/1/2 for single-class
        self.DFM_quality = [int(x) for x in self.config.DFM_quality.split(',')]  # e.g., "1" -> [1]
        self.skip_empty_filter = True
        annotations_dir = self.config.annotations_dir

        # Build image index once
        image_index = self._build_image_index(
            images_dir=data_dir,
            prefer_visualisation=getattr(self.config, "visualisation_type", None),
        )

        self.images.clear()
        self.masks.clear()

        for mask_filename in os.listdir(annotations_dir):
            if not mask_filename.lower().endswith((".tif", ".tiff")):
                continue

            key = self._two_part_prefix(mask_filename)
            if not key:
                # Skip masks that don't follow 'A__B__...' naming
                continue

            image_path = image_index.get(key)
            if not image_path:
                # No matching image for this key
                continue

            mask_path = os.path.join(annotations_dir, mask_filename)

            # Since preprocessing guarantees positives, just append
            self.images.append(image_path)
            self.masks.append(mask_path)

        if not self.images:
            raise RuntimeError(
                "No image/mask pairs loaded. Check directory paths and filename prefixes."
            )

    # ---------- data access ----------

    def __getitem__(self, index):
        # --- Read image (C,H,W) -> (H,W,C) ---
        with rasterio.open(self.images[index]) as src:
            image = src.read()  # (C, H, W)
        if image.shape[0] == 1:
            image = np.repeat(image, 3, axis=0)
        image = np.transpose(image, (1, 2, 0))  # (H, W, C)

        # --- Read/construct mask (binary 0/1) ---
        mask01 = self.process_single_mask(self.masks[index])  # (H, W) uint8 {0,1}

        # --- One-hot to match labels order ---
        mask_oh = np.stack([mask01 == 0, mask01 == 1], axis=-1).astype("float32")  # (H, W, 2)

        return self.apply_transformations(image, mask_oh)


In [27]:
index = 0

print(train_dataset.images[index])

with rasterio.open(train_dataset.images[index]) as src:
    image = src.read()  # (C, H, W)

if image.shape[0] == 1:
    image = np.repeat(image, 3, axis=0)

print(image.shape)

image = np.transpose(image, (1, 2, 0))  # (H, W, C)

print(image.shape)

# --- Read/construct mask (binary 0/1) ---
print(train_dataset.masks[index])
mask01 = process_single_mask(train_dataset.masks[index])  # (H, W) uint8 {0,1}  # TODO: Do te vrstice štima, tukaj pade! Kar moram dobiti je maska!

print(mask01.shape)

2025-11-11 21:23:40,598 WARNING CPLE_AppDefined in PROJ: proj_create_from_database: Cannot find proj.db
2025-11-11 21:23:40,599 WARNING CPLE_AppDefined in The definition of projected CRS EPSG:32634 got from GeoTIFF keys is not the same as the one from the EPSG registry, which may cause issues during reprojection operations. Set GTIFF_SRS_SOURCE configuration option to EPSG to use official parameters (overriding the ones from GeoTIFF keys), or to GEOKEYS to use custom values from GeoTIFF keys and drop the EPSG code.


r:\delovno\nejc\training_samples_BiH_v3\train\images\261248_4748416__BiH_ALS_2025_DMO_05m_slrm__images.tif
(3, 512, 512)
(512, 512, 3)
r:\delovno\nejc\training_samples_BiH_v3\train\segmentation_masks\261248_4748416__BiH_ALS_2025_DMO_05m_slrm__segmentation_masks.tif


2025-11-11 21:23:40,697 WARNING CPLE_AppDefined in PROJ: proj_create_from_database: Cannot find proj.db
2025-11-11 21:23:40,698 WARNING CPLE_AppDefined in The definition of projected CRS EPSG:32634 got from GeoTIFF keys is not the same as the one from the EPSG registry, which may cause issues during reprojection operations. Set GTIFF_SRS_SOURCE configuration option to EPSG to use official parameters (overriding the ones from GeoTIFF keys), or to GEOKEYS to use custom values from GeoTIFF keys and drop the EPSG code.


TypeError: getattr(): attribute name must be string

In [ ]:
def __getitem__(self, index):
    # --- Read image (C,H,W) -> (H,W,C) ---
    with rasterio.open(self.images[index]) as src:
        image = src.read()  # (C, H, W)
    if image.shape[0] == 1:
        image = np.repeat(image, 3, axis=0)
    image = np.transpose(image, (1, 2, 0))  # (H, W, C)

    # --- Read/construct mask (binary 0/1) ---
    mask01 = self.process_single_mask(self.masks[index])  # (H, W) uint8 {0,1}

    # --- One-hot to match labels order ---
    mask_oh = np.stack([mask01 == 0, mask01 == 1], axis=-1).astype("float32")  # (H, W, 2)

    return self.apply_transformations(image, mask_oh)

In [26]:
def process_single_mask(mask_path: str) -> np.ndarray:
    """
    Convert a multi-band TIFF mask into a binary 2D array (H, W) with values {0,1}.
    - If self.object_class == 'AO': union (logical OR) of up to first 3 class bands.
    - Else: read only band (self.object_class_band_id + 1).
    - Keep only pixels whose DFM value is in self.DFM_quality (e.g., [1]).
    """
    with rasterio.open(mask_path) as src:
        if getattr("object_class", None) == "AO":
            nb = min(3, src.count)
            arr = src.read(indexes=list(range(1, nb + 1)))         # (B, H, W)
            mask_bool = np.isin(arr, [1]).any(axis=0) # (H, W) bool
        else:
            band = src.read(0 + 1)         # (H, W)
            mask_bool = np.isin(band, [1])            # (H, W) bool

    return mask_bool.astype(np.uint8)  # {0,1}

### Loading train, validation and test data

Input parameters for train, validation and test data:

- **batch_size**: The number of samples processed before the model is updated. A larger batch size can speed up processing but requires more memory.
- **num_workers**: The number of worker processes that will be used for processing data. Increasing the number of workers can significantly speed up data processing, however, it also increases memory and CPU/GPU usage.
- **object_class**: A parameter that specifies the type of archaeological object you are interested in processing, e.g., 'AO', 'barrow', 'enclosure', 'ringfort'.
- **object_class_band_id**: An integer parameter identifying the band where the annotations for a specific object class are located within the segmentation masks.
- **visualisation_type**: The vizuelization type used for the patches, e.g., 'SLRM'.
- **DFM_quality**: List of annotation qualities to be included in the processed data, e.g., '1,2'.
- **keep_empty_patches**: A boolean parameter that controls if empty patches are kept. Set to False when training since "empty" data can't be used for training. For testing or validation, True can be used to check how the model handles empty patches.
- **shuffle**: Determines whether the data should be shuffled before being processed. 
- **data_dir**: The directory path where the input data is stored. 
- **annotations_dir**: The directory path where the segmentation masks are stored. 
- **transforms**: A list of transformations applied to the input data during processing.
- **target_transforms**: A list of transformations applied to the segmentation masks during processing.
- **joint_transforms**: Transformations applied simultaneously to both the input data and segmentation masks.

In [4]:
batch_size = 16
num_workers = 4
object_class = "barrow"
object_class_band_id = 1
visualisation_type = "SLRM"

In [5]:
train_data = r"r:\delovno\nejc\training_samples_BiH_v3\train\images"
train_mask = r"r:\delovno\nejc\training_samples_BiH_v3\train\segmentation_masks"

validation_data = r"r:\delovno\nejc\training_samples_BiH_v3\validation\images"
validation_mask = r"r:\delovno\nejc\training_samples_BiH_v3\validation\segmentation_masks"

test_data = r"r:\delovno\nejc\training_samples_BiH_v3\test\images"
test_mask = r"r:\delovno\nejc\training_samples_BiH_v3\test\segmentation_masks"

In [7]:
model_config = TiiLIDARDatasetSegmentation.get_fixed_model_config()

In [8]:
train_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1',
    "shuffle": True,
    "keep_empty_patches": False,
    "data_dir": train_data,
    "annotations_dir": train_mask,
    "joint_transforms": ["aitlas.transforms.FlipHVRandomRotate"],
    "transforms": ["aitlas.transforms.Transpose"],
	"target_transforms": ["aitlas.transforms.Transpose"]
}
train_dataset = TiiLIDARDatasetSegmentation(train_dataset_config)

validation_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1',
    "shuffle": False,
    "keep_empty_patches": False,
    "data_dir": validation_data,
    "annotations_dir": validation_mask,
    "transforms": ["aitlas.transforms.Transpose"],
    "target_transforms": ["aitlas.transforms.Transpose"]
}
validation_dataset = TiiLIDARDatasetSegmentation(validation_dataset_config)

test_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1',
    "shuffle": False,
    "keep_empty_patches": False,
    "data_dir": test_data,
    "annotations_dir": test_mask,
    "transforms": ["aitlas.transforms.Transpose"],
	"target_transforms": ["aitlas.transforms.Transpose"]
}
test_dataset = TiiLIDARDatasetSegmentation(test_dataset_config)

len(train_dataset), len(validation_dataset), len(test_dataset)

(4383, 708, 931)

In [9]:
train_dataset.images[0]

'r:\\delovno\\nejc\\training_samples_BiH_v3\\train\\images\\261248_4748416__BiH_ALS_2025_DMO_05m_slrm__images.tif'

### Model creation

In [7]:
model = HRNet(model_config)
model.prepare()

2025-11-11 20:56:40,168 INFO Loading pretrained weights from Hugging Face hub (timm/hrnet_w48.ms_in1k)
2025-11-11 20:56:40,665 INFO HTTP Request: HEAD https://huggingface.co/timm/hrnet_w48.ms_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2025-11-11 20:56:40,677 INFO [timm/hrnet_w48.ms_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


### Loading pretrained ADAF model (optional)

If you don't want to use an existing model, you can skip this step. If you do want to use one, set the model path, uncomment the lines, and run the cell to load the model into memory.

In [9]:
model_path = r".\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar" 
model.load_model(model_path)

2025-11-11 20:57:07,591 INFO Loading checkpoint .\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar
2025-11-11 20:57:09,837 INFO Loaded checkpoint .\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar at epoch 25


(25,
 0.013847780594127303,
 1698762193,
 'train_barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_Transformation')

### Training the model

Input parameters: 
- **epochs**: The total number of training cycles the model will undergo. Each epoch represents one complete pass of the training dataset through the model.
- **model_directory**: Path to the directory where the trained model and its checkpoints will be saved. This is used for storing the model during and after training.
- **run_id**: Name of the subdirectory within the model_directory to store results from different runs

In [11]:
epochs = 20
model_directory = r"r:\delovno\nejc\models\semantic_segmentation"
run_id = 'barrow_stone_v3'

In [ ]:
model.train_and_evaluate_model(
    train_dataset=train_dataset,
    val_dataset=validation_dataset,
    epochs=epochs,
    model_directory=model_directory,
    run_id=run_id
)

2025-11-11 20:58:32,623 INFO Starting training.
training:   0%|                                                                                | 0/274 [00:00<?, ?it/s]

### Model evaluation

In [ ]:
model = HRNet(model_config)
model.prepare()
model.running_metrics.reset()
model_path = r"./models/semantic_segmentation/best_checkpoint_test.pth.tar" # update the path!
model.evaluate(dataset=test_dataset, model_path=model_path)
model.running_metrics.get_scores(model.metrics)